# Notebook 06 — Next-Match Targets and Historical Features

## Objective
Create leakage-free supervised learning datasets for predicting
IPL players' next-match batting and bowling performance.

## Input datasets
- batting_player_match_regular.csv
- bowling_player_match_regular.csv

## Target variables
Batting:
- target_next_match_runs
- target_next_match_balls

Bowling:
- target_next_match_wickets
- target_next_match_runs_conceded

## Feature engineering
- Lag features
- Rolling averages (3, 5, 10 matches)
- Expanding historical averages
- Match recency
- Career-to-date statistics

## Validation
- Strict chronological ordering
- No current-match information in historical features
- No future information leakage
- Correct next-match target alignment

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PLAYER_MATCH_DIR = (
    PROJECT_ROOT / "data" / "processed" / "player_match"
)

print("Project root:", PROJECT_ROOT)
print("Input directory:", PLAYER_MATCH_DIR)
print("Directory exists:", PLAYER_MATCH_DIR.is_dir())

assert PLAYER_MATCH_DIR.is_dir()

Project root: D:\Git\Personal\ipl-player-performance-v2
Input directory: D:\Git\Personal\ipl-player-performance-v2\data\processed\player_match
Directory exists: True


In [2]:
input_files = {
    "batting": PLAYER_MATCH_DIR / "batting_player_match_regular.csv",
    "bowling": PLAYER_MATCH_DIR / "bowling_player_match_regular.csv",
}

for name, path in input_files.items():
    print(
        f"{name}: exists={path.is_file()}",
        f"| size={path.stat().st_size / 1024:.2f} KB"
        if path.is_file() else ""
    )

assert all(path.is_file() for path in input_files.values())

print("\nINPUT FILE VALIDATION: PASS")

batting: exists=True | size=1797.95 KB
bowling: exists=True | size=1121.13 KB

INPUT FILE VALIDATION: PASS


In [3]:
batting = pd.read_csv(
    input_files["batting"],
    dtype={"match_id": "string"},
    low_memory=False
)

bowling = pd.read_csv(
    input_files["bowling"],
    dtype={"match_id": "string"},
    low_memory=False
)

# Parse match dates.
batting["date"] = pd.to_datetime(
    batting["date"], errors="coerce"
)

bowling["date"] = pd.to_datetime(
    bowling["date"], errors="coerce"
)

print("Batting dataset:", batting.shape)
print("Bowling dataset:", bowling.shape)

Batting dataset: (18822, 17)
Bowling dataset: (14700, 16)


In [4]:
for name, df in {
    "BATTING": batting,
    "BOWLING": bowling
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    display(
        df.isna().sum()
        .to_frame("missing_count")
        .query("missing_count > 0")
    )

    print("\nSample records:")
    display(df.head(5))


BATTING

Columns:
['match_id', 'player', 'is_super_over', 'batting_team', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'innings_batted', 'not_out_innings', 'season', 'date', 'match_type']

Data types:
match_id                      string
player                           str
is_super_over                   bool
batting_team                     str
runs_scored                    int64
balls_faced                    int64
fours                          int64
sixes                          int64
dismissals                     int64
retired_hurt_count             int64
strike_rate                  float64
batting_average              float64
innings_batted                 int64
not_out_innings                int64
season                           str
date                  datetime64[us]
match_type                       str
dtype: object

Missing values:


,missing_count
strike_rate,59
batting_average,4167



Sample records:


,match_id,player,is_super_over,batting_team,runs_scored,balls_faced,fours,sixes,dismissals,retired_hurt_count,strike_rate,batting_average,innings_batted,not_out_innings,season,date,match_type
0,1082591,A Choudhary,False,Royal Challengers Bangalore,6,2,0,1,0,0,300.000000,NaN,1,1,2017,2017-04-05,T20
1,1082591,BCJ Cutting,False,Sunrisers Hyderabad,16,6,0,2,0,0,266.666667,NaN,1,1,2017,2017-04-05,T20
2,1082591,CH Gayle,False,Royal Challengers Bangalore,32,21,2,3,1,0,152.380952,32.0,1,0,2017,2017-04-05,T20
3,1082591,DA Warner,False,Sunrisers Hyderabad,14,8,2,1,1,0,175.000000,14.0,1,0,2017,2017-04-05,T20
4,1082591,DJ Hooda,False,Sunrisers Hyderabad,16,12,0,1,0,0,133.333333,NaN,1,1,2017,2017-04-05,T20



BOWLING

Columns:
['match_id', 'player', 'is_super_over', 'runs_conceded', 'legal_balls', 'wides_conceded', 'noballs_conceded', 'innings_bowled', 'wickets', 'overs_bowled', 'economy_rate', 'bowling_average', 'bowling_strike_rate', 'season', 'date', 'match_type']

Data types:
match_id                       string
player                            str
is_super_over                    bool
runs_conceded                   int64
legal_balls                     int64
wides_conceded                  int64
noballs_conceded                int64
innings_bowled                  int64
wickets                         int64
overs_bowled                  float64
economy_rate                  float64
bowling_average               float64
bowling_strike_rate           float64
season                            str
date                   datetime64[us]
match_type                        str
dtype: object

Missing values:


,missing_count
economy_rate,1
bowling_average,6251
bowling_strike_rate,6251



Sample records:


,match_id,player,is_super_over,runs_conceded,legal_balls,wides_conceded,noballs_conceded,innings_bowled,wickets,overs_bowled,economy_rate,bowling_average,bowling_strike_rate,season,date,match_type
0,1082591,A Choudhary,False,55,24,3,1,1,1,4.000000,13.750000,55.0,24.0,2017,2017-04-05,T20
1,1082591,A Nehra,False,42,24,1,0,1,2,4.000000,10.500000,21.0,12.0,2017,2017-04-05,T20
2,1082591,B Kumar,False,27,24,1,0,1,2,4.000000,6.750000,13.5,12.0,2017,2017-04-05,T20
3,1082591,BCJ Cutting,False,35,22,2,0,1,0,3.666667,9.545455,NaN,NaN,2017,2017-04-05,T20
4,1082591,Bipul Sharma,False,4,6,0,0,1,1,1.000000,4.000000,4.0,6.0,2017,2017-04-05,T20


In [5]:
required_batting = {
    "match_id",
    "player",
    "date",
    "season",
    "runs_scored",
    "balls_faced",
    "fours",
    "sixes",
    "strike_rate",
    "dismissals",
    "innings_batted",
    "not_out_innings",
}

required_bowling = {
    "match_id",
    "player",
    "date",
    "season",
    "runs_conceded",
    "legal_balls",
    "wickets",
    "economy_rate",
    "bowling_average",
    "bowling_strike_rate",
}

assert required_batting.issubset(batting.columns), (
    required_batting - set(batting.columns)
)

assert required_bowling.issubset(bowling.columns), (
    required_bowling - set(bowling.columns)
)

print("BATTING SCHEMA: PASS")
print("BOWLING SCHEMA: PASS")

BATTING SCHEMA: PASS
BOWLING SCHEMA: PASS


In [6]:
assert batting["player"].notna().all()
assert bowling["player"].notna().all()

assert batting["date"].notna().all()
assert bowling["date"].notna().all()

assert not batting.duplicated(
    ["match_id", "player"]
).any()

assert not bowling.duplicated(
    ["match_id", "player"]
).any()

print("Batting unique player-match keys:", len(batting))
print("Bowling unique player-match keys:", len(bowling))

print("\nPLAYER-MATCH KEY VALIDATION: PASS")

Batting unique player-match keys: 18822
Bowling unique player-match keys: 14700

PLAYER-MATCH KEY VALIDATION: PASS


In [7]:
print("BATTING DATE RANGE")
print("Start:", batting["date"].min())
print("End:", batting["date"].max())

print("\nBOWLING DATE RANGE")
print("Start:", bowling["date"].min())
print("End:", bowling["date"].max())

print("\nUnique batting dates:", batting["date"].nunique())
print("Unique bowling dates:", bowling["date"].nunique())

print("\nBatting seasons:")
print(sorted(batting["season"].dropna().unique()))

print("\nBowling seasons:")
print(sorted(bowling["season"].dropna().unique()))

BATTING DATE RANGE
Start: 2008-04-18 00:00:00
End: 2026-05-31 00:00:00

BOWLING DATE RANGE
Start: 2008-04-18 00:00:00
End: 2026-05-31 00:00:00

Unique batting dates: 947
Unique bowling dates: 947

Batting seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Bowling seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


In [10]:
# Identify players appearing in multiple matches on the same date.

batting_same_day = (
    batting.groupby(["player", "date"], as_index=False)
    .agg(
        match_count=("match_id", "nunique"),
        match_ids=("match_id", lambda x: list(x.unique()))
    )
    .query("match_count > 1")
    .reset_index(drop=True)
)

bowling_same_day = (
    bowling.groupby(["player", "date"], as_index=False)
    .agg(
        match_count=("match_id", "nunique"),
        match_ids=("match_id", lambda x: list(x.unique()))
    )
    .query("match_count > 1")
    .reset_index(drop=True)
)

print(
    "Batting player-date combinations with multiple matches:",
    len(batting_same_day)
)

print(
    "Bowling player-date combinations with multiple matches:",
    len(bowling_same_day)
)

display(batting_same_day.head(10))
display(bowling_same_day.head(10))

Batting player-date combinations with multiple matches: 0
Bowling player-date combinations with multiple matches: 0


,player,date,match_count,match_ids


,player,date,match_count,match_ids


In [13]:
# Same-day audit has already confirmed zero conflicts.
if batting_same_day.empty and bowling_same_day.empty:
    print("No same-day player-match conflicts found.")
    print("Chronological ordering by date is safe for each player.")

else:
    print("Same-day conflicts require additional inspection.")

    same_day_ids = set(
        batting_same_day["match_ids"].explode().dropna().astype(str)
    )

    matches_metadata = pd.read_csv(
        PROJECT_ROOT / "data" / "processed" / "matches.csv",
        dtype={"match_id": "string"},
        low_memory=False
    )

    matches_metadata["date"] = pd.to_datetime(
        matches_metadata["date"], errors="coerce"
    )

    same_day_metadata = matches_metadata.loc[
        matches_metadata["match_id"].isin(same_day_ids),
        [
            "match_id", "date", "season",
            "event_name", "event_match_number",
            "event_stage", "team1", "team2"
        ]
    ].sort_values(["date", "match_id"])

    display(same_day_metadata)

No same-day player-match conflicts found.
Chronological ordering by date is safe for each player.


In [14]:
# Sort each player's records chronologically.
batting = batting.sort_values(
    ["player", "date", "match_id"]
).reset_index(drop=True)

bowling = bowling.sort_values(
    ["player", "date", "match_id"]
).reset_index(drop=True)

# Verify ordering.
assert not batting.duplicated(["player", "date"]).any()
assert not bowling.duplicated(["player", "date"]).any()

print("Batting chronological ordering: PASS")
print("Bowling chronological ordering: PASS")

display(
    batting[
        ["player", "date", "match_id", "runs_scored"]
    ].head(10)
)

display(
    bowling[
        ["player", "date", "match_id", "wickets"]
    ].head(10)
)

Batting chronological ordering: PASS
Bowling chronological ordering: PASS


,player,date,match_id,runs_scored
0,A Ashish Reddy,2012-04-29,548346,10
1,A Ashish Reddy,2012-05-04,548352,3
2,A Ashish Reddy,2012-05-08,548359,8
3,A Ashish Reddy,2012-05-18,548373,10
4,A Ashish Reddy,2012-05-20,548376,4
5,A Ashish Reddy,2013-04-05,598000,7
6,A Ashish Reddy,2013-04-07,598004,14
7,A Ashish Reddy,2013-04-09,598048,3
8,A Ashish Reddy,2013-04-12,598010,16
9,A Ashish Reddy,2013-04-14,598013,4


,player,date,match_id,wickets
0,A Ashish Reddy,2012-04-26,548341,2
1,A Ashish Reddy,2012-04-29,548346,1
2,A Ashish Reddy,2012-05-01,548348,1
3,A Ashish Reddy,2012-05-04,548352,1
4,A Ashish Reddy,2012-05-06,548356,1
5,A Ashish Reddy,2012-05-08,548359,2
6,A Ashish Reddy,2012-05-10,548329,0
7,A Ashish Reddy,2012-05-18,548373,0
8,A Ashish Reddy,2012-05-20,548376,3
9,A Ashish Reddy,2013-04-05,598000,1


In [15]:
def create_next_match_dataset(
    df,
    player_col,
    date_col,
    target_columns,
    rolling_windows=(3, 5, 10)
):
    """
    Create next-appearance targets and leakage-free
    historical features for each player.
    """

    data = df.copy()

    data = data.sort_values(
        [player_col, date_col, "match_id"]
    ).reset_index(drop=True)

    grouped = data.groupby(player_col, sort=False)

    # Next-appearance targets.
    for col in target_columns:
        data[f"target_next_match_{col}"] = (
            grouped[col].shift(-1)
        )

    # Previous appearance date and days since previous appearance.
    data["previous_match_date"] = (
        grouped[date_col].shift(1)
    )

    data["days_since_previous_match"] = (
        data[date_col] - data["previous_match_date"]
    ).dt.days

    # Historical features: shift first, then calculate statistics.
    for col in target_columns:

        previous_values = grouped[col].shift(1)

        data[f"previous_{col}"] = previous_values

        for window in rolling_windows:
            data[f"{col}_rolling_mean_{window}"] = (
                previous_values
                .groupby(data[player_col])
                .transform(
                    lambda s: s.rolling(
                        window=window,
                        min_periods=1
                    ).mean()
                )
            )

        data[f"{col}_expanding_mean"] = (
            previous_values
            .groupby(data[player_col])
            .transform(
                lambda s: s.expanding(
                    min_periods=1
                ).mean()
            )
        )

    # First appearance has no previous-match history.
    data["matches_played_before"] = (
        data.groupby(player_col).cumcount()
    )

    return data

In [17]:
batting_features = create_next_match_dataset(
    df=batting,
    player_col="player",
    date_col="date",
    target_columns=[
        "runs_scored",
        "balls_faced",
        "fours",
        "sixes",
        "strike_rate",
        "dismissals"
    ],
    rolling_windows=(3, 5, 10)
)

print("Batting feature dataset:", batting_features.shape)

display(
    batting_features[
        [
            "player",
            "date",
            "runs_scored",
            "previous_runs_scored",
            "runs_scored_rolling_mean_3",
            "runs_scored_rolling_mean_5",
            "runs_scored_rolling_mean_10",
            "target_next_match_runs_scored",
            "target_next_match_balls_faced"
        ]
    ].head(15)
)

Batting feature dataset: (18822, 56)


,player,date,runs_scored,previous_runs_scored,runs_scored_rolling_mean_3,runs_scored_rolling_mean_5,runs_scored_rolling_mean_10,target_next_match_runs_scored,target_next_match_balls_faced
0,A Ashish Reddy,2012-04-29,10,NaN,NaN,NaN,NaN,3.0,3.0
1,A Ashish Reddy,2012-05-04,3,10.0,10.000000,10.00,10.000000,8.0,8.0
2,A Ashish Reddy,2012-05-08,8,3.0,6.500000,6.50,6.500000,10.0,4.0
3,A Ashish Reddy,2012-05-18,10,8.0,7.000000,7.00,7.000000,4.0,4.0
4,A Ashish Reddy,2012-05-20,4,10.0,7.000000,7.75,7.750000,7.0,4.0
5,A Ashish Reddy,2013-04-05,7,4.0,7.333333,7.00,7.000000,14.0,12.0
6,A Ashish Reddy,2013-04-07,14,7.0,7.000000,6.40,7.000000,3.0,4.0
7,A Ashish Reddy,2013-04-09,3,14.0,8.333333,8.60,8.000000,16.0,9.0
8,A Ashish Reddy,2013-04-12,16,3.0,8.000000,7.60,7.375000,4.0,5.0
9,A Ashish Reddy,2013-04-14,4,16.0,11.000000,8.80,8.333333,19.0,15.0


In [18]:
bowling_features = create_next_match_dataset(
    df=bowling,
    player_col="player",
    date_col="date",
    target_columns=[
        "wickets",
        "runs_conceded",
        "legal_balls",
        "economy_rate"
    ],
    rolling_windows=(3, 5, 10)
)

print("Bowling feature dataset:", bowling_features.shape)

display(
    bowling_features[
        [
            "player",
            "date",
            "wickets",
            "previous_wickets",
            "wickets_rolling_mean_3",
            "wickets_rolling_mean_5",
            "wickets_rolling_mean_10",
            "target_next_match_wickets",
            "target_next_match_runs_conceded"
        ]
    ].head(15)
)

Bowling feature dataset: (14700, 43)


,player,date,wickets,previous_wickets,wickets_rolling_mean_3,wickets_rolling_mean_5,wickets_rolling_mean_10,target_next_match_wickets,target_next_match_runs_conceded
0,A Ashish Reddy,2012-04-26,2,NaN,NaN,NaN,NaN,1.0,11.0
1,A Ashish Reddy,2012-04-29,1,2.0,2.000000,2.000000,2.000000,1.0,32.0
2,A Ashish Reddy,2012-05-01,1,1.0,1.500000,1.500000,1.500000,1.0,16.0
3,A Ashish Reddy,2012-05-04,1,1.0,1.333333,1.333333,1.333333,1.0,36.0
4,A Ashish Reddy,2012-05-06,1,1.0,1.000000,1.250000,1.250000,2.0,39.0
5,A Ashish Reddy,2012-05-08,2,1.0,1.000000,1.200000,1.200000,0.0,29.0
6,A Ashish Reddy,2012-05-10,0,2.0,1.333333,1.200000,1.333333,0.0,17.0
7,A Ashish Reddy,2012-05-18,0,0.0,1.000000,1.000000,1.142857,3.0,25.0
8,A Ashish Reddy,2012-05-20,3,0.0,0.666667,0.800000,1.000000,1.0,21.0
9,A Ashish Reddy,2013-04-05,1,3.0,1.000000,1.200000,1.222222,1.0,7.0


In [20]:
# Retain only records having a next-appearance target.

batting_model_data = batting_features.dropna(
    subset=["target_next_match_runs_scored"]
).copy()

bowling_model_data = bowling_features.dropna(
    subset=["target_next_match_wickets"]
).copy()

print("Batting rows before:", len(batting_features))
print("Batting rows with next-match targets:", len(batting_model_data))

print("\nBowling rows before:", len(bowling_features))
print("Bowling rows with next-match targets:", len(bowling_model_data))

Batting rows before: 18822
Batting rows with next-match targets: 18083

Bowling rows before: 14700
Bowling rows with next-match targets: 14123


In [22]:
# ==========================================
# TARGET ALIGNMENT AND LEAKAGE VALIDATION
# ==========================================

# 1. Batting target alignment
batting_check = batting_features.groupby("player", sort=False)

expected_batting_runs = batting_check["runs_scored"].shift(-1)
expected_batting_balls = batting_check["balls_faced"].shift(-1)

assert np.allclose(
    batting_features["target_next_match_runs_scored"].to_numpy(dtype=float),
    expected_batting_runs.to_numpy(dtype=float),
    equal_nan=True
)

assert np.allclose(
    batting_features["target_next_match_balls_faced"].to_numpy(dtype=float),
    expected_batting_balls.to_numpy(dtype=float),
    equal_nan=True
)

# 2. Bowling target alignment
bowling_check = bowling_features.groupby("player", sort=False)

expected_bowling_wickets = bowling_check["wickets"].shift(-1)
expected_bowling_runs = bowling_check["runs_conceded"].shift(-1)

assert np.allclose(
    bowling_features["target_next_match_wickets"].to_numpy(dtype=float),
    expected_bowling_wickets.to_numpy(dtype=float),
    equal_nan=True
)

assert np.allclose(
    bowling_features["target_next_match_runs_conceded"].to_numpy(dtype=float),
    expected_bowling_runs.to_numpy(dtype=float),
    equal_nan=True
)

# 3. Historical feature leakage validation
expected_previous_batting_runs = (
    batting_features.groupby("player", sort=False)["runs_scored"].shift(1)
)

expected_previous_bowling_wickets = (
    bowling_features.groupby("player", sort=False)["wickets"].shift(1)
)

assert np.allclose(
    batting_features["previous_runs_scored"].to_numpy(dtype=float),
    expected_previous_batting_runs.to_numpy(dtype=float),
    equal_nan=True
)

assert np.allclose(
    bowling_features["previous_wickets"].to_numpy(dtype=float),
    expected_previous_bowling_wickets.to_numpy(dtype=float),
    equal_nan=True
)

print("Batting next-match target alignment: PASS")
print("Bowling next-match target alignment: PASS")
print("Historical feature shift validation: PASS")

print("\nTARGET AND LEAKAGE VALIDATION: PASS")

Batting next-match target alignment: PASS
Bowling next-match target alignment: PASS
Historical feature shift validation: PASS

TARGET AND LEAKAGE VALIDATION: PASS


In [23]:
# Create output directory
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Save complete feature datasets
batting_features.to_csv(
    FEATURE_DIR / "batting_next_match_features.csv",
    index=False
)

bowling_features.to_csv(
    FEATURE_DIR / "bowling_next_match_features.csv",
    index=False
)

# Save supervised model datasets
batting_model_data.to_csv(
    FEATURE_DIR / "batting_model_dataset.csv",
    index=False
)

bowling_model_data.to_csv(
    FEATURE_DIR / "bowling_model_dataset.csv",
    index=False
)

print("FEATURE DATASETS SAVED")

for file in FEATURE_DIR.glob("*.csv"):
    print(f"{file.name}: {file.stat().st_size / 1024:.2f} KB")

FEATURE DATASETS SAVED
batting_model_dataset.csv: 7589.56 KB
batting_next_match_features.csv: 7841.99 KB
bowling_model_dataset.csv: 4071.09 KB
bowling_next_match_features.csv: 4212.79 KB


In [24]:
assert (FEATURE_DIR / "batting_next_match_features.csv").exists()
assert (FEATURE_DIR / "bowling_next_match_features.csv").exists()
assert (FEATURE_DIR / "batting_model_dataset.csv").exists()
assert (FEATURE_DIR / "bowling_model_dataset.csv").exists()

print("FEATURE EXPORT VALIDATION: PASS")

FEATURE EXPORT VALIDATION: PASS


In [25]:
# Inspect target columns and feature columns.

for name, df in {
    "BATTING": batting_model_data,
    "BOWLING": bowling_model_data
}.items():

    target_columns = [
        col for col in df.columns
        if col.startswith("target_next_match_")
    ]

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Target columns:", target_columns)
    print("Target missing values:")
    print(df[target_columns].isna().sum())

    print("\nPotential identifier columns:")
    print([
        col for col in df.columns
        if col in ["match_id", "player", "date", "season"]
    ])


BATTING
Shape: (18083, 56)
Target columns: ['target_next_match_runs_scored', 'target_next_match_balls_faced', 'target_next_match_fours', 'target_next_match_sixes', 'target_next_match_strike_rate', 'target_next_match_dismissals']
Target missing values:
target_next_match_runs_scored     0
target_next_match_balls_faced     0
target_next_match_fours           0
target_next_match_sixes           0
target_next_match_strike_rate    55
target_next_match_dismissals      0
dtype: int64

Potential identifier columns:
['match_id', 'player', 'season', 'date']

BOWLING
Shape: (14123, 43)
Target columns: ['target_next_match_wickets', 'target_next_match_runs_conceded', 'target_next_match_legal_balls', 'target_next_match_economy_rate']
Target missing values:
target_next_match_wickets          0
target_next_match_runs_conceded    0
target_next_match_legal_balls      0
target_next_match_economy_rate     1
dtype: int64

Potential identifier columns:
['match_id', 'player', 'season', 'date']


In [26]:
# ==========================================
# FEATURE LEAKAGE AUDIT
# ==========================================

def audit_model_columns(df, dataset_name):

    target_cols = [
        c for c in df.columns
        if c.startswith("target_next_match_")
    ]

    identifier_cols = [
        c for c in [
            "match_id", "player", "date"
        ]
        if c in df.columns
    ]

    metadata_cols = [
        c for c in [
            "season", "match_type", "batting_team",
            "is_super_over"
        ]
        if c in df.columns
    ]

    historical_cols = [
        c for c in df.columns
        if (
            c.startswith("previous_")
            or "_rolling_mean_" in c
            or c.endswith("_expanding_mean")
            or c in [
                "matches_played_before",
                "days_since_previous_match"
            ]
        )
    ]

    print(f"\n{'=' * 55}")
    print(dataset_name)
    print("=" * 55)

    print("Target columns:", target_cols)
    print("Identifier columns:", identifier_cols)
    print("Metadata columns:", metadata_cols)
    print("Historical feature columns:", len(historical_cols))

    assert len(target_cols) > 0
    assert not any(
        c.startswith("target_next_match_")
        for c in historical_cols
    )

    print("\nInitial column audit: PASS")


audit_model_columns(batting_model_data, "BATTING")
audit_model_columns(bowling_model_data, "BOWLING")


BATTING
Target columns: ['target_next_match_runs_scored', 'target_next_match_balls_faced', 'target_next_match_fours', 'target_next_match_sixes', 'target_next_match_strike_rate', 'target_next_match_dismissals']
Identifier columns: ['match_id', 'player', 'date']
Metadata columns: ['season', 'match_type', 'batting_team', 'is_super_over']
Historical feature columns: 33

Initial column audit: PASS

BOWLING
Target columns: ['target_next_match_wickets', 'target_next_match_runs_conceded', 'target_next_match_legal_balls', 'target_next_match_economy_rate']
Identifier columns: ['match_id', 'player', 'date']
Metadata columns: ['season', 'match_type', 'is_super_over']
Historical feature columns: 23

Initial column audit: PASS


In [27]:
# Candidate features exclude all future targets and identifiers.

batting_excluded = {
    c for c in batting_model_data.columns
    if c.startswith("target_next_match_")
} | {"match_id", "player", "date"}

bowling_excluded = {
    c for c in bowling_model_data.columns
    if c.startswith("target_next_match_")
} | {"match_id", "player", "date"}

X_batting = batting_model_data.drop(
    columns=list(batting_excluded)
)

X_bowling = bowling_model_data.drop(
    columns=list(bowling_excluded)
)

print("Batting candidate feature shape:", X_batting.shape)
print("Bowling candidate feature shape:", X_bowling.shape)

assert not any(
    c.startswith("target_next_match_")
    for c in X_batting.columns
)

assert not any(
    c.startswith("target_next_match_")
    for c in X_bowling.columns
)

print("\nTARGET EXCLUSION VALIDATION: PASS")

Batting candidate feature shape: (18083, 47)
Bowling candidate feature shape: (14123, 36)

TARGET EXCLUSION VALIDATION: PASS


In [28]:
# ==========================================
# FINAL FEATURE AVAILABILITY AUDIT
# ==========================================

def inspect_unclassified_features(
    df,
    dataset_name,
    metadata_columns
):
    excluded = {
        c for c in df.columns
        if c.startswith("target_next_match_")
    }

    excluded.update(["match_id", "player", "date"])
    excluded.update(metadata_columns)

    historical = {
        c for c in df.columns
        if (
            c.startswith("previous_")
            or "_rolling_mean_" in c
            or c.endswith("_expanding_mean")
            or c in [
                "matches_played_before",
                "days_since_previous_match"
            ]
        )
    }

    unclassified = [
        c for c in df.columns
        if c not in excluded and c not in historical
    ]

    print(f"\n{'=' * 55}")
    print(dataset_name)
    print("=" * 55)
    print("Unclassified/current-match feature candidates:")
    for col in unclassified:
        print(f"- {col}")

    return unclassified


batting_unclassified = inspect_unclassified_features(
    batting_model_data,
    "BATTING",
    ["season", "match_type", "batting_team", "is_super_over"]
)

bowling_unclassified = inspect_unclassified_features(
    bowling_model_data,
    "BOWLING",
    ["season", "match_type", "is_super_over"]
)


BATTING
Unclassified/current-match feature candidates:
- runs_scored
- balls_faced
- fours
- sixes
- dismissals
- retired_hurt_count
- strike_rate
- batting_average
- innings_batted
- not_out_innings

BOWLING
Unclassified/current-match feature candidates:
- runs_conceded
- legal_balls
- wides_conceded
- noballs_conceded
- innings_bowled
- wickets
- overs_bowled
- economy_rate
- bowling_average
- bowling_strike_rate


In [29]:
# ==========================================
# APPROVED FEATURE SCHEMA
# ==========================================

batting_target_cols = [
    c for c in batting_model_data.columns
    if c.startswith("target_next_match_")
]

bowling_target_cols = [
    c for c in bowling_model_data.columns
    if c.startswith("target_next_match_")
]

identifier_cols = ["match_id", "player", "date"]

batting_feature_cols = [
    c for c in batting_model_data.columns
    if c not in batting_target_cols + identifier_cols
]

bowling_feature_cols = [
    c for c in bowling_model_data.columns
    if c not in bowling_target_cols + identifier_cols
]

# Ensure no target columns enter feature lists.
assert not any(
    c.startswith("target_next_match_")
    for c in batting_feature_cols
)

assert not any(
    c.startswith("target_next_match_")
    for c in bowling_feature_cols
)

# Check that historical features exist.
assert "previous_runs_scored" in batting_feature_cols
assert "previous_wickets" in bowling_feature_cols

print("Approved batting features:", len(batting_feature_cols))
print("Approved bowling features:", len(bowling_feature_cols))

print("\nFEATURE SCHEMA VALIDATION: PASS")

Approved batting features: 47
Approved bowling features: 36

FEATURE SCHEMA VALIDATION: PASS


In [30]:
import json

feature_schema = {
    "prediction_definition": (
        "Predict a player's next regular-innings appearance "
        "after the current match has completed."
    ),
    "batting": {
        "identifier_columns": identifier_cols,
        "feature_columns": batting_feature_cols,
        "target_columns": batting_target_cols
    },
    "bowling": {
        "identifier_columns": identifier_cols,
        "feature_columns": bowling_feature_cols,
        "target_columns": bowling_target_cols
    }
}

schema_path = FEATURE_DIR / "feature_schema.json"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(feature_schema, file, indent=4)

print("Saved:", schema_path)

Saved: D:\Git\Personal\ipl-player-performance-v2\data\processed\features\feature_schema.json
